In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

In [2]:
import bitsandbytes

In [3]:
torch.cuda.is_available()

True

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.float16)

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

d:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VICTUS\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [6]:
for param in model.parameters():
  param.requires_grad = False  # freeze the model - train adapters later

model.gradient_checkpointing_enable()  # reduce number of stored activations
model.enable_input_require_grads()

In [7]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [8]:
tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token',
 'additional_special_tokens']

In [9]:
model.to(device)
def model_test(text):
    inputs = tokenizer(text,
            padding=True,
            truncation=True,
            return_tensors='pt').to(device)
    output = model.generate(**inputs,
                            max_new_tokens=30)
    print(tokenizer.batch_decode(output)[0])
    
    
    

In [10]:
model_test("""## instruction: you are a bot
           ## user: who are you
           ## assistent:""")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


## instruction: you are a bot
           ## user: who are you
           ## assistent: what are you
           ## user: what is your name
           ## assistent: what is your name
           ## user: what is your


In [11]:
ds = load_dataset("tatsu-lab/alpaca")

In [12]:
def get_intruct_text(batch):
    batch['text'] = batch['text'].split('\n\n', maxsplit=1)[-1]
    return batch

In [29]:
data_pd

,instruction,input,output,text
0,<instruction> Give three tips for staying heal...,<instruction> Give three tips for staying heal...,<response> 1.Eat a balanced diet and make sure...,Below is an instruction that describes a task....
1,<instruction> What are the three primary colors?,<instruction> What are the three primary color...,"<response> The three primary colors are red, b...",Below is an instruction that describes a task....
2,<instruction> Describe the structure of an atom.,<instruction> Describe the structure of an ato...,"<response> An atom is made up of a nucleus, wh...",Below is an instruction that describes a task....
3,<instruction> How can we reduce air pollution?,<instruction> How can we reduce air pollution?...,<response> There are a number of ways to reduc...,Below is an instruction that describes a task....
4,<instruction> Describe a time when you had to ...,<instruction> Describe a time when you had to ...,<response> I had to make a difficult decision ...,Below is an instruction that describes a task....
...,...,...,...,...
51997,<instruction> Generate an example of what a re...,<instruction> Generate an example of what a re...,"<response> Jean Tremaine\n1234 Main Street, An...",Below is an instruction that describes a task....
51998,<instruction> Arrange the items given below in...,<instruction> Arrange the items given below in...,<response> I eating cake.,"Below is an instruction that describes a task,..."
51999,<instruction> Write an introductory paragraph ...,<instruction> Write an introductory paragraph ...,<response> Michelle Obama is an inspirational ...,"Below is an instruction that describes a task,..."
52000,<instruction> Generate a list of five things o...,<instruction> Generate a list of five things o...,<response> 1. Research potential opportunities...,Below is an instruction that describes a task....


0        Below is an instruction that describes a task....
1        Below is an instruction that describes a task....
2        Below is an instruction that describes a task....
3        Below is an instruction that describes a task....
4        Below is an instruction that describes a task....
                               ...                        
51997    Below is an instruction that describes a task....
51998    Below is an instruction that describes a task,...
51999    Below is an instruction that describes a task,...
52000    Below is an instruction that describes a task....
52001    Below is an instruction that describes a task,...
Name: text, Length: 52002, dtype: object

In [17]:
ds = ds.map(get_intruct_text)

In [18]:
ds = ds.remove_columns([
    'instruction',
    'input',
    'output'
])

In [19]:
train_ds = ds['train'].train_test_split(test_size=0.2, seed=0)

In [20]:
val_test = train_ds['test'].train_test_split(.4, seed=42)

In [21]:
ds = DatasetDict({
    "train": train_ds['train'],
    "test": val_test['test'],
    "val": val_test['train']
})

In [22]:
print(ds['train'][3]['text'])

### Instruction:
Optimize this query for maximum recall:

### Input:
SELECT * FROM  table WHERE column1 = "value1"

### Response:
SELECT * FROM  table WHERE column1 LIKE "%value1%"


In [23]:
ds.set_format('pandas')
ds.reset_format()

In [24]:
def tokenize_data(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, return_tensors='pt')

In [25]:
final_ds = ds.map(tokenize_data, batched=True)

In [26]:
# def add_labels(batch):
#     batch['labels'] = batch['input_ids']
#     return batch

In [27]:
# ds = final_ds.map(add_labels)

In [28]:
ds = final_ds.remove_columns('text')

In [29]:
lora_cfg = LoraConfig(r=8,
                     lora_alpha=16,
                     target_modules=['q_proj', 'v_proj', "o_proj"],
                     lora_dropout=0.1,
                     task_type=TaskType.CAUSAL_LM)

In [30]:
model_peft = get_peft_model(model, lora_cfg)

In [31]:
torch.cuda.is_available()

True

In [32]:
import torch

In [33]:
torch.cuda.is_available()

True

In [34]:
!nvidia-smi

Mon Apr 28 17:09:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.02                 Driver Version: 576.02         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   39C    P8              4W /   30W |    1979MiB /   4096MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [47]:
trainer_args = TrainingArguments(
    output_dir='QwenVajiInstructModel_0.5B',
    max_steps=4000,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    save_steps=100,
    learning_rate=1e-4,
    eval_strategy='steps',
    eval_steps=300,
    fp16=True,
    disable_tqdm=False,
    logging_strategy='steps',   
    logging_steps=1,
    report_to='tensorboard'
 )

In [48]:
trainer_args.device

device(type='cuda', index=0)

In [49]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [50]:
trainer = Trainer(model=model_peft, 
                    args=trainer_args, 
                    train_dataset=ds['train'], 
                    eval_dataset=ds['val'],
                    data_collator=data_collator)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [51]:
ds['train']

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 41601
})

In [52]:
trainer.train(resume_from_checkpoint="QwenVajiInstructModel_0.5B/checkpoint-1000")

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [21]:
my_model = PeftModel.from_pretrained(model, "QwenVajiInstructModel_0.5B\\checkpoint-1000")

d:\Anaconda\Lib\site-packages\peft\tuners\tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [22]:
_ = my_model.to(device)

In [23]:
def text_formatter(context, input):
    text = f"""### Instruction:
{context}

### Input:
{input}

### Response:

"""
    return text
    

In [28]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=896, out_features=896, bias=True)
          (k_proj): Linear8bitLt(in_features=896, out_features=128, bias=True)
          (v_proj): Linear8bitLt(in_features=896, out_features=128, bias=True)
          (o_proj): Linear8bitLt(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear8bitLt(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear8bitLt(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), ep

In [1]:
input = text_formatter(context="""You are a chat assistend""", input='tell me a good joke')
inputs = tokenizer(input, return_tensors='pt').to(device)
output = model.generate(**inputs, max_new_tokens=1000)
print("Original Model Output:",tokenizer.batch_decode(output)[0])
print('-'*20)
print(f'lora model {tokenizer.batch_decode(my_model.generate(**inputs, max_new_tokens=1500))[0]}')


NameError: name 'text_formatter' is not defined

trainer.push_hub()

In [58]:
trainer.push_to_hub()

HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-680cbc62-4e1bc0754b08f10913cdf0dc;5d6b2a14-0874-4eff-ae85-d9fbe37a7de7)

Invalid username or password.